# Opisy kategorii — Obsidian Vault

Dla każdego pliku w `zurada_vault/Kategorie/` generuje krótki opis po polsku i wstawia go do pliku MD.

97 kategorii → ~5 requestów (batch 20).

In [1]:
import os, json, re, time
from pathlib import Path
from openai import OpenAI

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "Brak klucza API")  # ustaw zmienną środowiskową lub wklej tu
MODEL          = 'gpt-5.5'
KATEGORIE_DIR  = Path('zurada_vault/Kategorie')
BATCH_SIZE     = 20
SLEEP_BETWEEN  = 0.4
MAX_RETRIES    = 3

client = OpenAI(api_key=OPENAI_API_KEY)
files  = sorted(KATEGORIE_DIR.glob('*.md'))
print(f'Znaleziono {len(files)} plikow kategorii.')


Znaleziono 97 plikow kategorii.


In [2]:
def parse_product_count(text: str) -> int:
    m = re.search(r'## Produkty \((\d+)\)', text)
    return int(m.group(1)) if m else 0

def already_has_opis(text: str) -> bool:
    return '## Opis' in text

categories = []
for f in files:
    text  = f.read_text(encoding='utf-8')
    slug  = f.stem
    count = parse_product_count(text)
    categories.append({'file': f, 'slug': slug, 'count': count, 'text': text})

todo = [c for c in categories if not already_has_opis(c['text'])]
print(f'Do opisania: {len(todo)} kategorii.')
print('Przyklad slug:', categories[0]['slug'], '| produktow:', categories[0]['count'])


Do opisania: 97 kategorii.
Przyklad slug: chemia-samochodowa | produktow: 24


In [3]:
SYSTEM_PROMPT = (
    'Jestes ekspertem ds. chemii czyszczecej i profesjonalnych srodkow higieny.\n'
    'Opisujesz kategorie produktow czyszczacych dla systemu rekomendacji.\n\n'
    'Dla kazdej kategorii napisz KROTKI opis (2-3 zdania) po polsku:\n'
    '- Kim sa odbiorcy tej kategorii lub jakie sa zastosowania\n'
    '- Jakie problemy / potrzeby adresuje ta kategoria produktow\n'
    '- Jesli nazwa jest skrotem branżowym (np. horeca, CIP, mycie-cisnieniowe) — wyjasnic co oznacza\n'
    '- Pisz zwiezle, konkretnie, bez ogolnikow\n\n'
    'Zwroc wylacznie JSON: {"descriptions": {"slug": "opis", ...}}'
)

def describe_batch(batch: list) -> dict:
    lines = [
        f'- "{c["slug"]}" ({c["count"]} produktow)'
        for c in batch
    ]
    user_msg = (
        f'Opisz {len(batch)} kategorii produktow czyszczacych Zurada:\n\n'
        + '\n'.join(lines)
        + '\n\nZwroc JSON: {"descriptions": {"slug": "opis", ...}}'
    )
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': user_msg},
                ],
                response_format={'type': 'json_object'},
                max_completion_tokens=2000,
            )
            content = resp.choices[0].message.content
            if not content:
                raise ValueError('Pusta odpowiedz')
            return json.loads(content).get('descriptions', {})
        except Exception as e:
            wait = 2 ** attempt
            print(f'  [RETRY {attempt}/{MAX_RETRIES}] {e} — czekam {wait}s')
            time.sleep(wait)
    return {}

print('Funkcja describe_batch gotowa.')


Funkcja describe_batch gotowa.


In [4]:
def insert_opis(text: str, slug: str, opis: str) -> str:
    """Wstawia sekcje ## Opis po naglowku # slug."""
    header = f'# {slug}'
    idx = text.find(header)
    if idx == -1:
        return text
    insert_pos = idx + len(header)
    opis_block = f'\n\n## Opis\n\n{opis}'
    return text[:insert_pos] + opis_block + text[insert_pos:]


batches = [todo[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
updated = 0

for i, batch in enumerate(batches, 1):
    slugs = [c['slug'] for c in batch]
    print(f'Batch {i}/{len(batches)}: {slugs[0]} ... {slugs[-1]}')

    descriptions = describe_batch(batch)

    for cat in batch:
        slug  = cat['slug']
        opis  = descriptions.get(slug, '').strip()
        if not opis:
            print(f'  [WARN] brak opisu dla: {slug}')
            continue
        new_text = insert_opis(cat['text'], slug, opis)
        cat['file'].write_text(new_text, encoding='utf-8')
        updated += 1

    time.sleep(SLEEP_BETWEEN)

print(f'\nZaktualizowano {updated}/{len(todo)} plikow.')


Batch 1/5: chemia-samochodowa ... higiena-powierzchni-catering
Batch 2/5: higiena-powierzchni-firmy-sprzatajace ... mycie-felg-i-kolpakow
Batch 3/5: mycie-podlog-maszynowe-catering ... ochrona-osuszanie-i-nablyszczanie-pojazdow
Batch 4/5: odkamienianie-firmy-sprzatajace ... przemysl-farmaceutyczny
Batch 5/5: przemysl-kosmetyczny ... zabezpieczanie-i-pielegnacja-podlog

Zaktualizowano 97/97 plikow.


In [5]:
# Podglad — sprawdz losowy plik
import random
sample = random.choice(files)
print(f'=== {sample.name} ===')
print(sample.read_text(encoding='utf-8')[:600])


=== udraznianie-rur-i-odplywow-instytucje.md ===
---
typ: kategoria
liczba_produktow: 1
---

# udraznianie-rur-i-odplywow-instytucje

## Opis

Środki do udrażniania odpływów przeznaczone dla szkół, urzędów, placówek medycznych i innych obiektów publicznych. Pomagają szybko reagować na niedrożne umywalki, kratki ściekowe i zlewy, zmniejszając przestoje w użytkowaniu sanitariatów i zapleczy.

## Produkty (1)

- [[Produkty/Zurada Przepływ|Zurada Przepływ]]

